# 3. Cycles / Loops

A graph edge can point back to a node that already ran — LangGraph doesn't
require a DAG. Demonstrated with a retry loop: ask the model for a one-sentence
description, and if it comes back longer than `max_words`, loop back and ask
again (up to `max_attempts`), instead of accepting the first answer unconditionally.

**A real, verified finding**: feeding the *previous* attempt's text and word
count back into the retry prompt is what makes convergence reliable — without
that feedback, `llama3.2` doesn't reliably converge on a stricter word limit
even after several retries. This notebook includes that feedback; the Playground
below has you try removing it.

**Prerequisites:** Ollama running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph

from models.chat_models.ollama_models import SupportedModel, get_chat_model


class RefineState(TypedDict):
    topic: str
    max_words: int
    max_attempts: int
    attempt: int
    text: str
    word_count: int


llm = get_chat_model(SupportedModel.llama3_2)


def generate(state: RefineState) -> dict:
    feedback = ""
    if state["attempt"] > 0:
        feedback = (
            f' Your previous attempt ("{state["text"]}") was {state["word_count"]} '
            "words — too long. Try again, shorter."
        )
    response = llm.invoke(
        f"Describe {state['topic']} in {state['max_words']} words or fewer. "
        "Respond with ONLY the description, no preamble." + feedback
    )
    text = response.content.strip()
    return {"text": text, "word_count": len(text.split()), "attempt": state["attempt"] + 1}


def should_continue(state: RefineState) -> Literal["generate", "__end__"]:
    if state["word_count"] <= state["max_words"] or state["attempt"] >= state["max_attempts"]:
        return END
    return "generate"


graph = StateGraph(RefineState)
graph.add_node("generate", generate)
graph.add_edge(START, "generate")
graph.add_conditional_edges("generate", should_continue)
compiled = graph.compile()

In [ ]:
result = compiled.invoke(
    {"topic": "the water cycle", "max_words": 8, "max_attempts": 5, "attempt": 0, "text": "", "word_count": 0}
)
print(f"text ({result['word_count']} words, {result['attempt']} attempt(s)):")
print(result["text"])
print("met target:", result["word_count"] <= 8)

## 🧪 Playground

**1. Remove the feedback** — build a second `generate` node that ignores `state["attempt"]`/`state["text"]`/`state["word_count"]` entirely (always the same prompt). Does it converge as reliably within `max_attempts`?

In [ ]:
# TODO: define generate_no_feedback, build a second graph with it, compare attempts_used across a few topics


**2. A stricter limit** — try `max_words=4` and see how many attempts it needs.

In [ ]:
# TODO: compiled.invoke with max_words=4


**3. Lower `max_attempts`** to 2 and see what happens when it can't converge in time — does it just give up with an over-length answer?

In [ ]:
# TODO: compiled.invoke with max_attempts=2 and a hard topic
